# RetainFlow - Train Churn CatBoost

Notebook d'entrainement detaille: chargement PostgreSQL, preprocessing, split, entrainement CatBoost, evaluation, SHAP, MLflow et sauvegarde des predictions.

## 1. Imports


In [ ]:
%load_ext autoreload
%autoreload 2

import json

from retainflow.config import load_churn_model_config
from retainflow.logging import get_logger
from retainflow.tracking.runtime import configure_local_mlflow_runtime

configure_local_mlflow_runtime()

import matplotlib.pyplot as plt
import mlflow
import mlflow.catboost
import pandas as pd
from catboost import CatBoostClassifier, Pool

from retainflow.data.dataset import ChurnDatasetLoader
from retainflow.data.splitting import TemporalDatasetSplitter
from retainflow.evaluation.metrics import BinaryClassifierEvaluator, ConfusionMatrixReporter
from retainflow.evaluation.visualization import ClassDistributionPlotter
from retainflow.features.engineering import ChurnFeatureEngineer
from retainflow.features.preprocessing import FEATURE_COLUMNS, ChurnPreprocessor
from retainflow.pipelines.train_churn import (
    ChurnModelTrainer,
    configure_mlflow,
    log_mlflow_dataset,
    log_mlflow_provenance,
    predict_positive_class_probability,
    save_predictions,
    shap_version,
)
from retainflow.tracking import launch_mlflow_ui

In [ ]:
logger = get_logger("retainflow.notebooks.train_churn")
logger.info("Training notebook started")

## 2. Charger La Configuration


In [ ]:
config = load_churn_model_config("config/churn_model.yml")
config

## 3. MLflow Central


### 3.1 Configurer Le Tracking Et Le Registry


In [ ]:
tracking_uri = configure_mlflow(config)
logger.info("MLflow tracking URI actif: %s", tracking_uri)

mlflow_state = {
    "tracking_uri": tracking_uri,
    "registry_uri": mlflow.get_registry_uri(),
    "experiment_name": config.experiment_name,
}
mlflow_state


### 3.2 Lancer L'Interface MLflow


In [ ]:
mlflow_ui = launch_mlflow_ui(config)
mlflow_ui


## 4. Charger Les Donnees Depuis PostgreSQL


In [ ]:
loader = ChurnDatasetLoader(config)
raw_dataset = loader.load()
print(f"Raw dataset with {raw_dataset.shape[0]} rows and {raw_dataset.shape[1]} columns loaded.")

In [ ]:
display(raw_dataset.head(5))

## 5. Controler Les Splits Et Le Label


In [ ]:
label_distribution = (
    raw_dataset.groupby("split_name")
    .agg(rows=("customer_id", "count"), churn_rate=("churn_label", "mean"))
    .reset_index()
)
label_distribution

## 6. Visualiser La Distribution Des Classes


In [ ]:
class_distribution_plotter = ClassDistributionPlotter(size=(15, 6))
class_distribution = class_distribution_plotter.distribution_frame(raw_dataset)
class_distribution

In [ ]:
class_distribution_plot_path = config.class_distribution_plot_path
class_distribution_ax = class_distribution_plotter.plot(raw_dataset, path=class_distribution_plot_path)
plt.show()
class_distribution_plot_path

## 7. Feature Engineering Metier


In [ ]:
feature_engineer = ChurnFeatureEngineer()
feature_dataset = feature_engineer.transform(raw_dataset)
feature_dataset[
    [
        "observation_date",
        "customer_id",
        "churn_label",
        "churn_date",
        "customer_lifecycle_status",
        "customer_age_years",
        "customer_lifetime_days",
        "premium_per_policy",
        "renewal_window",
        "days_to_churn_after_observation",
    ]
].head()

In [ ]:
feature_dataset[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False).head(15)

## 8. Split Temporel Avant Preprocessing


In [ ]:
splitter = TemporalDatasetSplitter()
raw_splits = splitter.split(feature_dataset)
splitter.class_distribution(feature_dataset)

## 9. Fit Preprocessor Sur Train Puis Transformer Chaque Split


In [ ]:
preprocessor = ChurnPreprocessor()
preprocessor.fit(raw_splits["train"].data)

dataset = pd.concat(
    [preprocessor.transform(split.data) for split in raw_splits.values()],
    ignore_index=True,
)
missing_after_preprocessing = dataset[FEATURE_COLUMNS].isna().sum().sort_values(ascending=False)
missing_after_preprocessing[missing_after_preprocessing > 0]

## 10. Declarer Les Features CatBoost


In [ ]:
cat_features = preprocessor.catboost_feature_indices()
feature_contract = pd.DataFrame({"feature": FEATURE_COLUMNS, "is_categorical": [i in cat_features for i in range(len(FEATURE_COLUMNS))]})
feature_contract

## 11. Split Train / Validation / Test / Backtest


In [ ]:
splits = splitter.split(dataset)
train_split = splits["train"]
valid_split = splits["validation"]
test_split = splits["test"]
backtest_split = splits["backtest"]

train_data, train_x, train_y = train_split.data, train_split.features, train_split.target
valid_data, valid_x, valid_y = valid_split.data, valid_split.features, valid_split.target
test_data, test_x, test_y = test_split.data, test_split.features, test_split.target
backtest_data, backtest_x, backtest_y = backtest_split.data, backtest_split.features, backtest_split.target

pd.DataFrame(
    [
        ("train", len(train_data), train_y.mean(), train_split.time.min(), train_split.time.max()),
        ("validation", len(valid_data), valid_y.mean(), valid_split.time.min(), valid_split.time.max()),
        ("test", len(test_data), test_y.mean(), test_split.time.min(), test_split.time.max()),
        ("backtest", len(backtest_data), backtest_y.mean(), backtest_split.time.min(), backtest_split.time.max()),
    ],
    columns=["split_name", "rows", "churn_rate", "time_min", "time_max"],
)

In [ ]:
train_data[["observation_date", "customer_id", "split_name", "churn_label"]].head()

## 12. Creer Les Pools CatBoost


In [ ]:
train_pool = Pool(train_x, label=train_y, cat_features=cat_features)
valid_pool = Pool(valid_x, label=valid_y, cat_features=cat_features)
test_pool = Pool(test_x, label=test_y, cat_features=cat_features)
backtest_pool = Pool(backtest_x, label=backtest_y, cat_features=cat_features)

train_pool.num_row(), train_pool.num_col()

## 13. Initialiser Le Modele


In [ ]:
model = CatBoostClassifier(
    iterations=config.iterations,
    learning_rate=config.learning_rate,
    depth=config.depth,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=config.random_seed,
    verbose=False,
    allow_writing_files=False,
)
model

## 14. Creer Le Trainer


In [ ]:
evaluator = BinaryClassifierEvaluator(threshold=config.prediction_threshold)
trainer = ChurnModelTrainer(config=config, model=model, evaluator=evaluator)
trainer

## 15. Entrainer Et Logger Le Run


In [ ]:
active_run = mlflow.active_run()
if active_run is not None:
    logger.info("Fermeture du run MLflow actif avant nouveau run: %s", active_run.info.run_id)
    mlflow.end_run()

run = mlflow.start_run(
    run_name="retainflow_churn_catboost_postgres",
    tags={
        "project": "retainflow",
        "environment": "development",
        "stage": "model_training",
        "model_name": "CatBoostClassifier",
        "data_source": "postgresql",
    },
    description="CatBoost churn model trained from PostgreSQL customer 360 snapshots.",
    log_system_metrics=config.mlflow_log_system_metrics,
)
run_id = run.info.run_id
logger.info("MLflow run started: %s", run_id)
log_mlflow_dataset(dataset, config)
log_mlflow_provenance()
mlflow.log_params(
    {
        "model_type": "CatBoostClassifier",
        "feature_table": config.feature_fqn,
        "label_table": config.label_fqn,
        "iterations": config.iterations,
        "learning_rate": config.learning_rate,
        "depth": config.depth,
        "prediction_threshold": config.prediction_threshold,
        "random_seed": config.random_seed,
        "shap_version": shap_version(),
    }
)
mlflow.log_table(
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "is_categorical": [index in cat_features for index in range(len(FEATURE_COLUMNS))],
        }
    ),
    "tables/feature_contract.json",
)
mlflow.log_table(class_distribution, "tables/class_distribution_by_split.json")
mlflow.log_figure(class_distribution_ax.figure, "figures/class_distribution_by_split.png")

trainer.fit(train_pool, valid_pool)
run_id

## 16. Tracer Les Courbes Train / Eval


In [ ]:
training_curve = trainer.training_curve()
training_curve.tail()

In [ ]:
curve_metric = "Logloss"
curve_data = training_curve.loc[training_curve["metric"] == curve_metric]
ax = curve_data.pivot(index="iteration", columns="dataset", values="value").plot(figsize=(10, 5))
ax.set_title(f"CatBoost {curve_metric} - train vs eval")
ax.set_xlabel("iteration")
ax.set_ylabel(curve_metric)
plt.show()

In [ ]:
training_curve_path = config.training_curve_path
training_curve_path.parent.mkdir(parents=True, exist_ok=True)
training_curve.to_csv(training_curve_path, index=False)
mlflow.log_table(training_curve, "tables/catboost_training_curve.json")
training_curve_path

## 17. Evaluer Validation / Test / Backtest


In [ ]:
metrics_by_split, probabilities_by_split = trainer.evaluate(
    pools_by_split={
        "validation": valid_pool,
        "test": test_pool,
        "backtest": backtest_pool,
    },
    targets_by_split={
        "validation": valid_y,
        "test": test_y,
        "backtest": backtest_y,
    },
)

for split_name, metrics in metrics_by_split.items():
    for metric_name, value in metrics.items():
        mlflow.log_metric(f"{split_name}_{metric_name}", value)

metrics_frame = pd.DataFrame(metrics_by_split).T.reset_index(names="split_name")
mlflow.log_table(metrics_frame, "tables/metrics_by_split.json")
metrics_frame

In [ ]:
mlflow.log_metric("train_rows", len(train_x))
mlflow.log_metric("validation_rows", len(valid_x))
mlflow.log_metric("test_rows", len(test_x))
mlflow.log_metric("backtest_rows", len(backtest_x))

## 18. Matrice De Confusion


In [ ]:
confusion_reporter = ConfusionMatrixReporter(
    threshold=config.prediction_threshold,
    size=(15, 6),
)
confusion_matrix_frame = confusion_reporter.matrix_frame(
    targets_by_split={
        "test": test_y,
        "backtest": backtest_y,
    },
    probabilities_by_split=probabilities_by_split,
)

confusion_matrix_table_path = config.confusion_matrix_table_path
confusion_matrix_plot_path = config.confusion_matrix_plot_path
confusion_matrix_table_path.parent.mkdir(parents=True, exist_ok=True)
confusion_matrix_frame.to_csv(confusion_matrix_table_path, index=False)
confusion_matrix_figure = confusion_reporter.plot(
    confusion_matrix_frame,
    path=confusion_matrix_plot_path,
)

mlflow.log_table(confusion_matrix_frame, "tables/confusion_matrix_by_split.json")
mlflow.log_figure(confusion_matrix_figure, "figures/confusion_matrix_by_split.png")

confusion_matrix_frame


## 19. Analyser La Distribution Des Scores


In [ ]:
score_distribution = []
for split_name, probabilities in probabilities_by_split.items():
    scores = pd.Series(probabilities)
    score_distribution.append(
        {
            "split_name": split_name,
            "min": scores.min(),
            "p50": scores.quantile(0.50),
            "p90": scores.quantile(0.90),
            "p95": scores.quantile(0.95),
            "p99": scores.quantile(0.99),
            "max": scores.max(),
            "predicted_positives": int((scores >= config.prediction_threshold).sum()),
        }
    )

pd.DataFrame(score_distribution)

## 20. Explicabilite SHAP


In [ ]:
shap_sample_size = 1000
shap_explainer = trainer.shap_explainer()
shap_summary = shap_explainer.summary_frame(train_pool, sample_size=shap_sample_size)
shap_summary.head(15)

In [ ]:
shap_path = config.shap_summary_path
shap_report_path = config.shap_agent_report_path
shap_plot_path = config.shap_feature_importance_plot_path

shap_explainer.save_summary_csv(shap_summary, shap_path)
mlflow.log_table(shap_summary, "tables/shap_summary.json")
shap_report = shap_explainer.build_agent_report(
    summary=shap_summary,
    metrics_by_split=metrics_by_split,
    model_name=config.registered_model_name,
    run_id=run_id,
    sample_size=shap_sample_size,
)
shap_explainer.save_agent_report(shap_report, shap_report_path)
shap_explainer.plot_feature_importance(shap_summary, shap_plot_path)

mlflow.log_dict(shap_report, "explainability/shap_agent_report.json")
mlflow.log_artifact(str(shap_plot_path), artifact_path="figures")

{
    "summary_csv": str(shap_path),
    "agent_json": str(shap_report_path),
    "plot_png": str(shap_plot_path),
}

In [ ]:
plot_image = plt.imread(shap_plot_path)
plt.figure(figsize=(10, 7))
plt.imshow(plot_image)
plt.axis("off")
plt.show()

In [ ]:
with shap_report_path.open(encoding="utf-8") as file:
    agent_explainability_payload = json.load(file)

agent_explainability_payload["top_features"][:10]

## 21. Logger Le Modele Dans MLflow


In [ ]:
mlflow.catboost.log_model(model, artifact_path="model")

## 22. Sauvegarder Les Predictions Dans PostgreSQL


In [ ]:
save_predictions(config, dataset, probabilities_by_split, run_id)

## 23. Fermer Le Run Et Resumer


In [ ]:
mlflow.end_run()
logger.info("Training complete: run_id=%s", run_id)

result = {
    "run_id": run_id,
    "tracking_uri": mlflow.get_tracking_uri(),
    "training_curve_path": str(training_curve_path),
    "class_distribution_plot_path": str(class_distribution_plot_path),
    "confusion_matrix_table_path": str(confusion_matrix_table_path),
    "confusion_matrix_plot_path": str(confusion_matrix_plot_path),
    "shap_report_path": str(shap_report_path),
    "shap_plot_path": str(shap_plot_path),
    "metrics": metrics_by_split,
    "shap_top_features": shap_summary.head(10).to_dict(orient="records"),
}
result